<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator — Google Colab（LM Link）

ColabでLM Studio（llmster）を動かし、LM Link経由でPCと接続するのじゃ。認証情報は`dbv4-wd14-tagger`と共通のGoogleドライブフォルダーへ**暗号化保存**する。モデルはpfnet公式のGGUFファイルだけを使用し、既定は8B／Q4_K_Mじゃ。

## 1. サーバー設定

`dbv4-wd14-tagger`のColab版と同じ`USE_GOOGLE_DRIVE`と`DRIVE_FOLDER_NAME`を使用する。既定の保存先は `MyDrive/.colab-secret/` じゃ。

In [ ]:
#@title ⚙️ サーバー設定 (Configuration)
#@markdown dbv4-wd14-taggerと同じGoogleドライブ保存先設定じゃ。

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = ".colab-secret" #@param {type:"string"}

CONFIG_DRIVE_FOLDER = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else ".colab-secret"
if (
    CONFIG_DRIVE_FOLDER in (".", "..")
    or "/" in CONFIG_DRIVE_FOLDER
    or "\\" in CONFIG_DRIVE_FOLDER
):
    raise ValueError("Googleドライブの保存先は単一フォルダー名を指定するのじゃ")
print(
    "[INFO] Googleドライブ保存: "
    + (f"有効 (MyDrive/{CONFIG_DRIVE_FOLDER})" if USE_GOOGLE_DRIVE else "無効")
)

## 2. LM Studioのインストール

先にインストールし、次のセルで認証情報を復元してからデーモンを起動する。

In [ ]:
import os
import subprocess
from pathlib import Path

subprocess.run(["pkill", "-f", "llmster"], check=False)
subprocess.run(["pkill", "-f", "lmstudio"], check=False)
subprocess.run(["bash", "-c", "curl -fsSL https://lmstudio.ai/install.sh | bash"], check=True)
os.environ["PATH"] = str(Path.home() / ".lmstudio" / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["lms", "bootstrap"], check=True)
subprocess.run(["sync"], check=True)
print("LM Studioのセットアップが完了したのじゃ。")

## 3. Googleドライブへのログイン状態保存

LM Studioは環境によって `~/.lmstudio` または `~/.cache/lm-studio` に認証状態を保存するため、両方から認証ファイルとLM Linkの端末識別情報を取得する。DBV4のTailscaleと同様、Googleドライブの `MyDrive/.colab-secret/lmstudio/` に保存し、次回起動時に復元する。パスフレーズは不要じゃ。

認証状態の保存先が見つからなくてもログイン済みセッションの実行は止めない。保存できなかった場合は警告を出すため、次回起動時の自動再認証省略は保証されないのじゃ。

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

LM_HOME = Path.home() / ".lmstudio"
LM_CACHE_HOME = Path.home() / ".cache" / "lm-studio"
DRIVE_AUTH_DIR = None
AUTH_DIRS = ("credentials",)
IDENTITY_FILES = (".internal/local-identity.json", ".internal/user-profile.json")

if USE_GOOGLE_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_AUTH_DIR = Path("/content/drive/MyDrive") / CONFIG_DRIVE_FOLDER / "lmstudio"

def auth_roots():
    return (("lmstudio", LM_HOME), ("lm-studio-cache", LM_CACHE_HOME))

def copy_login_files(source, destination):
    files = []
    for directory in AUTH_DIRS:
        root = source / directory
        if root.is_dir() and not root.is_symlink():
            files.extend(path for path in root.rglob("*") if path.is_file() and not path.is_symlink())
    files.extend(
        source / name for name in IDENTITY_FILES
        if (source / name).is_file() and not (source / name).is_symlink()
    )
    copied = 0
    for original in files:
        relative = original.relative_to(source)
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(original, target)
        copied += 1
    return copied

def restore_login():
    if not USE_GOOGLE_DRIVE:
        return False
    restored = 0
    for label, root in auth_roots():
        restored += copy_login_files(DRIVE_AUTH_DIR / label, root)
    if not restored:
        restored += copy_login_files(DRIVE_AUTH_DIR, LM_HOME)
    if restored:
        print(f"[INFO] GoogleドライブからLM Link認証状態を復元: {restored}ファイル")
    else:
        print("[INFO] 保存済みのLM Link認証状態なし。必要な場合は初回ログインするのじゃ。")
    return bool(restored)

def backup_login():
    if not USE_GOOGLE_DRIVE:
        return 0
    saved = 0
    for label, root in auth_roots():
        count = copy_login_files(root, DRIVE_AUTH_DIR / label)
        if count:
            print(f"[INFO] LM Studio状態を保存: {label} / {count}ファイル")
            saved += count
    if not saved:
        print(
            "[WARN] LM Studioはログイン状態でも、既知の保存先に認証ファイルがないのじゃ。"
            "バックアップできなかったが、LM Linkの処理は続行する。"
        )
        print(
            f"[INFO] 確認先: {LM_HOME / 'credentials'}、"
            f"{LM_CACHE_HOME / 'credentials'}"
        )
    return saved

auth_restored = restore_login()

## 4. LM Linkのログインと接続

保存済み認証情報が有効なら再ログインせずに接続する。初回またはログイン期限切れなら `lms login` の認証URLをブラウザで開くのじゃ。ログイン後に認証情報を暗号化保存する。

In [ ]:
import subprocess
import time

subprocess.run(["lms", "daemon", "up"], check=True)
time.sleep(3)
whoami = subprocess.run(
    ["lms", "whoami"], stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL, check=False, timeout=30
)
if whoami.returncode != 0:
    print("LM Studioのログインが必要じゃ。表示される認証URLで承認するのじゃ。")
    subprocess.run(["lms", "login"], check=True)
else:
    print("保存済みのLM Studioログインが有効じゃ。")
subprocess.run(["lms", "link", "enable"], check=True)
backup_login()
print("LM Linkが有効になったのじゃ。")

## 5. PLaMo 3 Translateモデルのダウンロードとロード

既定は**8B／Q4_K_M**。0.8B・2B・8B・31Bからサイズを選び、pfnetが公式に配布しているGGUFの量子化形式だけ指定する。`lms get`の検索処理を使わず、PLaMo公式のGGUFファイルを直接取得するため、問題のコマンドは実行しない。中断したダウンロードは途中から再開する。Colabランタイムを破棄するとモデルファイルは消えるが、Googleドライブの保存済み認証情報は残る。31BはGPUメモリ不足でロードできない場合があるのじゃ。

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import requests
from tqdm.auto import tqdm

plamo_size = "8b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0"]

OFFICIAL_GGUF_FORMATS = {
    "0.8b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "2b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "8b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
    "31b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
}
if plamo_size not in OFFICIAL_GGUF_FORMATS:
    raise ValueError("未対応のPLaMoサイズじゃ")
if quantization not in OFFICIAL_GGUF_FORMATS[plamo_size]:
    raise ValueError(f"pfnet公式で配布されていない組み合わせじゃ: {plamo_size}/{quantization}")

repo_name = f"plamo-3-translate-{plamo_size}-GGUF"
filename = f"plamo-3-translate-{plamo_size}-{quantization}.gguf"
model_dir = Path.home() / ".lmstudio" / "models" / "pfnet" / repo_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / filename
partial_path = model_dir / (filename + ".part")
model_id = f"plamo-3-translate-{plamo_size}-colab"
url = f"https://huggingface.co/pfnet/{repo_name}/resolve/main/{filename}?download=true"

def get_gguf_size(download_url):
    try:
        response = requests.head(download_url, allow_redirects=True, timeout=20)
        response.raise_for_status()
        size = response.headers.get("Content-Length") or response.headers.get("X-Linked-Size")
        return int(size) if size and int(size) > 0 else None
    except (requests.RequestException, TypeError, ValueError):
        print("[WARN] 総容量を取得できないため、ダウンロード済み容量と速度を表示するのじゃ。", flush=True)
        return None


def download_with_progress(download_url, destination):
    total_bytes = get_gguf_size(download_url)
    downloaded_bytes = destination.stat().st_size if destination.exists() else 0
    if total_bytes is not None and downloaded_bytes > total_bytes:
        raise RuntimeError("一時ファイルが公式GGUFの総容量を超えているのじゃ")
    command = [
        "curl", "--fail", "--location", "--silent", "--show-error",
        "--retry", "3", "--retry-delay", "3",
        "--continue-at", "-", "--output", str(destination), download_url
    ]
    with tqdm(
        total=total_bytes, initial=downloaded_bytes,
        unit="B", unit_scale=True, unit_divisor=1024,
        desc="PLaMo GGUF", dynamic_ncols=True, mininterval=0.5
    ) as progress:
        process = subprocess.Popen(command)
        try:
            while process.poll() is None:
                time.sleep(0.5)
                current_bytes = destination.stat().st_size if destination.exists() else 0
                if current_bytes != progress.n:
                    progress.update(current_bytes - progress.n)
            current_bytes = destination.stat().st_size if destination.exists() else 0
            if current_bytes != progress.n:
                progress.update(current_bytes - progress.n)
            exit_code = process.wait()
            if exit_code != 0:
                raise subprocess.CalledProcessError(exit_code, command)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                process.wait()
            raise
    if total_bytes is not None and destination.stat().st_size != total_bytes:
        raise RuntimeError("GGUFのダウンロード容量が公式ファイルと一致しないのじゃ")


if not model_path.exists() or model_path.stat().st_size < 100_000_000:
    print(f"公式GGUFを取得: pfnet/{repo_name}/{filename}", flush=True)
    download_with_progress(url, partial_path)
    if partial_path.stat().st_size < 100_000_000:
        raise RuntimeError("GGUFのダウンロード結果が小さすぎるのじゃ")
    os.replace(partial_path, model_path)
else:
    print(f"ダウンロード済みGGUFを再利用: {filename}", flush=True)

subprocess.run(["lms", "unload", "--all"], check=True)
subprocess.run(
    ["lms", "load", str(model_path), "--gpu", "max", "--identifier", model_id],
    check=True
)
subprocess.run(["lms", "server", "start", "--port", "1234"], check=True)
subprocess.run(["lms", "ps"], check=True)
print(f"ロード完了: {model_id}（LM Linkから利用可能）", flush=True)